In [2]:
import pandas as pd
import pingouin as pg
import krippendorff
import numpy as np

# ======================================================
# Read Excel file
# ======================================================

df = pd.read_excel("Human_validation.xlsx")

# ======================================================
# Define constructs
# ======================================================

constructs = {
    "Perceived Quality":
        ["PQ_R1","PQ_R2","PQ_R3","PQ_LLM"],

    "Perceived Value":
        ["PV_R1","PV_R2","PV_R3","PV_LLM"],

    "Perceived Image":
        ["PI_R1","PI_R2","PI_R3","PI_LLM"],

    "Confirmation of Expectations":
        ["CE_R1","CE_R2","CE_R3","CE_LLM"],

    "Overall Satisfaction":
        ["OS_R1","OS_R2","OS_R3","OS_LLM"],

    "Customer Trust":
        ["CT_R1","CT_R2","CT_R3","CT_LLM"],

    "Customer Complaints":
        ["CC_R1","CC_R2","CC_R3","CC_LLM"],

    "Customer Loyalty":
        ["CL_R1","CL_R2","CL_R3","CL_LLM"]
}

results = []

# ======================================================
# Loop through constructs
# ======================================================

for construct, cols in constructs.items():

    ###################################################
    # ICC(2,k)
    ###################################################

    temp = df[cols].copy()

    long = temp.reset_index().melt(
        id_vars="index",
        var_name="Rater",
        value_name="Score"
    )

    long.rename(columns={"index":"Target"}, inplace=True)

    icc = pg.intraclass_corr(
        data=long,
        targets="Target",
        raters="Rater",
        ratings="Score"
    )

    icc2k = icc.loc[icc['Type']=="ICC2k","ICC"].values[0]

    ###################################################
    # Krippendorff Alpha
    ###################################################

    human = df[cols[:3]].to_numpy().T

    alpha = krippendorff.alpha(
        reliability_data=human,
        level_of_measurement='interval'
    )

    ###################################################
    # Human-LLM agreement within ±20
    ###################################################

    llm = df[cols[3]]

    human_scores = df[cols[:3]]

    agreement = (
        np.abs(human_scores.sub(llm, axis=0)) <= 20
    )

    pct = agreement.mean().mean()*100

    ###################################################
    results.append([
        construct,
        round(icc2k,3),
        round(alpha,3),
        round(pct,1)
    ])

# ======================================================
# Results
# ======================================================

results = pd.DataFrame(results,
                       columns=[
                           "Construct",
                           "ICC(2,k)",
                           "Krippendorff α",
                           "% within ±20"
                       ])

print(results)

results.to_excel("Reliability_results.xlsx",
                 index=False)

                      Construct  ICC(2,k)  Krippendorff α  % within ±20
0             Perceived Quality     1.000           0.998         100.0
1               Perceived Value     0.982           0.980          94.7
2               Perceived Image     0.992           0.958          99.0
3  Confirmation of Expectations     0.990           0.949         100.0
4          Overall Satisfaction     0.995           0.973         100.0
5                Customer Trust     0.991           0.951          99.3
6           Customer Complaints     0.987           0.938         100.0
7              Customer Loyalty     0.994           0.972          99.7
